# Continuum Background Estimation

This example shows how to use cosipy for estimating the continuum background. For more details on how the algorithm works, see the other notebook in this directory. In short, the method is based on the traditional on-off analysis, where the background for a source region on the sky is estimated by performing some kind of interpolation of the nearby surrounding region. The main difference with a Compton telescope is that we are now performing the on-off analysis in the Compton data space.

In this particular example, we want to estimate the background for the Crab. We start with the full dataset. This contains the full background, which includes instrumental + astrophysical, where the latter consists of all astrophysical sources other than the Crab. Then, for each bin of Em and Phi, we mask the Crab in the PsiChi plane based on the point source response. Finally, we interpolate over the masked region using an inpainting method.   

The accuracy of this method depends strongly on two things:
1) Choosing the proper source region to mask
2) Making an accurate interpolation

The current source code takes a very simple appoach for both of these, but eventually they need to be improved. For the first point, ultimately we should use the ARM measurement, and we'll need to figure out the optimal percentage of counts to mask. A major challenge here is that point sources have really long tails in the ARM distribution, extending over the entire sky. So we probably can't use a typical confidence level of 95% for the masking. For the second point, we are currently using the simplest possible inpainting algorithm. More sophisticated methods are needed. The best alrorithm will likely come from deep convolution neural networks. An alterantive to inpainting methods is to use background templates, which can be scaled outside of the masked region. The code also needs to be developed in a way that will make this easy to do. Finally, the code is super slow and needs to be vectorized. 

In [ ]:
from cosipy.background_estimation import ContinuumEstimation
from cosipy.spacecraftfile import SpacecraftFile
from cosipy.util import fetch_wasabi_file
import os
import logging
import astropy.units as u
from astropy.coordinates import SkyCoord
from histpy import Histogram
import matplotlib.pyplot as plt
logging.basicConfig()
logging.getLogger().setLevel(logging.INFO)
%matplotlib inline

The notebook requires the following files:
1) DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth.ori
2) SMEXv12.Continuum.HEALPixO3_10bins_log_flat.binnedimaging.imagingresponse.nonsparse_nside8.area.good_chunks_unzip.earthocc.h5
3) crab_bkg_binned_data_galactic.hdf5
4) inputs_crab.yaml

They can be downloaded using the cells below.

In [ ]:
# 20280301_3_month_modifies.ori
fetch_wasabi_file('COSI-SMEX/DC3/Data/Orientation/DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth.ori')

In [ ]:
# Detector response file
# Make sure to unzip file after downloading!
fetch_wasabi_file('COSI-SMEX/DC2/Responses/SMEXv12.Continuum.HEALPixO3_10bins_log_flat.binnedimaging.imagingresponse.nonsparse_nside8.area.good_chunks_unzip.earthocc.h5.zip')

In [ ]:
# crab_bkg_binned_data_galactic.hdf5
fetch_wasabi_file('COSI-SMEX/cosipy_tutorials/background_estimation/crab_bkg_binned_data_galactic.hdf5')

In [ ]:
# inputs_crab.yaml
fetch_wasabi_file('COSI-SMEX/cosipy_tutorials/background_estimation/inputs_crab.yaml')

Define instance of class:

In [ ]:
instance = ContinuumEstimation()

In order to estimate the background, we need the point source response. If you don't already have this, you can calculate it, as shown below. Note that the coordinates of the Crab need to be passed as a tuple, giving Galactic longitude and latitude in degrees. 

In [ ]:
data_path = "/Users/parshad/Software/cosipyFiles"

# # Orientatin file:
# ori_file = os.path.join(data_path,"DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth.ori")

# # Spacecraft orientation:
# sc_orientation = SpacecraftFile.parse_from_file(ori_file)

# # Detector response:
# dr = os.path.join(data_path,\
#         "SMEXv12.Continuum.HEALPixO3_10bins_log_flat.binnedimaging.imagingresponse.nonsparse_nside8.area.good_chunks_unzip.earthocc.h5")

# # crab = SkyCoord(l=184.56*u.deg,b=-5.78*u.deg,frame="galactic")
# AGN4151 = SkyCoord(l=155.0770628414967*u.deg, b=75.0630654405054*u.deg,frame="galactic")
# psr = instance.calc_psr(sc_orientation, dr, AGN4151)
# psr.write(data_path+"/"+"AGN4151_psr.h5")

# Alternatively, you can load a PSR from file with: 
psr = instance.load_psr_from_file(data_path+"/"+"AGN4151_psr.h5")

Now let's calculate the estimated background. To make a short example, we'll only consider 1 Em bin and 2 Phi bins, as specified by the optional keywords e_loop and s_loop, respectively. We'll also make plots here for demonstrational purposes. 

Note that the current code has not yet been optimized for speed, as it uses a simple nested for loop. The time required to generate the estimated background using all bins is roughyly 4 hours. The option to use a subset of the Em bins and/or Phi bins may be useful for analyses that also use a given subset, but at this point the main motivation for this option is for demonstrational purposes, and when using this option, nothing is done with the other bins. 

In [ ]:
data_file = "/Users/parshad/Software/AGN_Coronae/SourceInj/NGC_4151_ec200_bkg_hist.hdf5"
data_yaml = "inputs_crab.yaml"
# estimated_bg = instance.continuum_bg_estimation(data_file, data_yaml, psr, make_plots=True, e_loop=(2,3), s_loop=(4,6))
estimated_bg = instance.continuum_bg_estimation(data_file, data_yaml, psr, containment=0.40, make_plots=True)

Finaly, let's save the estimated background to file:

In [ ]:
estimated_bg.write(data_path+"/"+"NGC4151_ec200_bkg_hist_estimated_background_containment0p60_allBins.hdf5",overwrite=True)

Plot the estimated and actual simulated background spectra

In [ ]:
sim_bkg = Histogram.open(data_path+"/"+"bkg_binned_data.hdf5").project("Em")
# est_bkg060 = Histogram.open(data_path+"/"+"NGC4151_ec200_bkg_hist_estimated_background_containment0p60_allBins.hdf5").project("Em")
NGC4151 = Histogram.open("/Users/parshad/Software/AGN_Coronae/SourceInj/NGC_4151_ec200_COSI_sourceInj_correctFlux_hist.hdf5").project("Em")
# NGC4151_bkg = Histogram.open("/Users/parshad/Software/AGN_Coronae/SourceInj/NGC_4151_ec200_bkg_hist.hdf5").project("Em")
# NGC4151_sub060 = (NGC4151_bkg - est_bkg060)
est_bkg_040 = Histogram.open(data_path+"/"+"NGC4151_ec200_bkg_hist_estimated_background_containment0p40_allBins.hdf5").project("Em")
# NGC4151_sub040 = NGC4151_bkg - est_bkg_040
# est_bkg_030 = Histogram.open(data_path+"/"+"NGC4151_ec200_bkg_hist_estimated_background_containment0p30_allBins.hdf5").project("Em")
# NGC4151_sub030 = NGC4151_bkg - est_bkg_030
# est_bkg_050 = Histogram.open(data_path+"/"+"NGC4151_ec200_bkg_hist_estimated_background_containment0p50_allBins.hdf5").project("Em")
# NGC4151_sub050 = NGC4151_bkg - est_bkg_050


fig, ax = plt.subplots(figsize=(12, 9))
sim_bkg.draw(ax, label="Simulated real background", color="green")
est_bkg_040.draw(ax, label="Estimated background for NGC 4151", color="orange", linestyle="dashed")
# NGC4151_bkg.draw(ax, label="NGC4151 ec200 plus background", color="red")
# NGC4151.draw(ax, label="Simulated NGC4151 ec200", color="blue")
# NGC4151_sub030.draw(ax, label="(NGC4151 ec200 plus simulated bkg) - Estimated background (Containment: 0.30)", color="green", linestyle="dashed")
# NGC4151_sub040.draw(ax, label="(NGC4151 ec200 plus simulated bkg) - Estimated background (Containment: 0.40)", color="red", linestyle="dashed")
# NGC4151_sub050.draw(ax, label="(NGC4151 ec200 plus simulated bkg) - Estimated background (Containment: 0.50)", color="orange", linestyle="dashed")
# NGC4151_sub060.draw(ax, label="(NGC4151 ec200 plus simulated bkg) - Estimated background (Containment: 0.60)", color="purple", linestyle="dashed")


ax.set_yscale("log")
ax.set_xscale("log")

# ax.set_ylim(-20000, 170000)
ax.set_ylabel("Counts")
ax.set_xlabel("Energy [keV]")

plt.legend(fontsize=12, frameon=False)